In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/wordsforthewise/lending-club/rejected_2007_to_2018Q4.csv.gz
/kaggle/input/datasets/wordsforthewise/lending-club/accepted_2007_to_2018Q4.csv.gz
/kaggle/input/datasets/wordsforthewise/lending-club/accepted_2007_to_2018q4.csv/accepted_2007_to_2018Q4.csv
/kaggle/input/datasets/wordsforthewise/lending-club/rejected_2007_to_2018q4.csv/rejected_2007_to_2018Q4.csv


In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("wordsforthewise/lending-club")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/wordsforthewise/lending-club


In [4]:
df_acc = pd.read_csv('/kaggle/input/datasets/wordsforthewise/lending-club/accepted_2007_to_2018Q4.csv.gz', low_memory = False)

In [5]:
df_rej = pd.read_csv('/kaggle/input/datasets/wordsforthewise/lending-club/rejected_2007_to_2018Q4.csv.gz', low_memory = False)

In [6]:
all_col = list(df_acc.columns)

In [8]:
all_col_rej = list(df_rej.columns)

In [9]:
all_col

['id',
 'member_id',
 'loan_amnt',
 'funded_amnt',
 'funded_amnt_inv',
 'term',
 'int_rate',
 'installment',
 'grade',
 'sub_grade',
 'emp_title',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'issue_d',
 'loan_status',
 'pymnt_plan',
 'url',
 'desc',
 'purpose',
 'title',
 'zip_code',
 'addr_state',
 'dti',
 'delinq_2yrs',
 'earliest_cr_line',
 'fico_range_low',
 'fico_range_high',
 'inq_last_6mths',
 'mths_since_last_delinq',
 'mths_since_last_record',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'initial_list_status',
 'out_prncp',
 'out_prncp_inv',
 'total_pymnt',
 'total_pymnt_inv',
 'total_rec_prncp',
 'total_rec_int',
 'total_rec_late_fee',
 'recoveries',
 'collection_recovery_fee',
 'last_pymnt_d',
 'last_pymnt_amnt',
 'next_pymnt_d',
 'last_credit_pull_d',
 'last_fico_range_high',
 'last_fico_range_low',
 'collections_12_mths_ex_med',
 'mths_since_last_major_derog',
 'policy_code',
 'application_type',
 'annual_inc_joint',
 '

In [10]:
all_col_rej

['Amount Requested',
 'Application Date',
 'Loan Title',
 'Risk_Score',
 'Debt-To-Income Ratio',
 'Zip Code',
 'State',
 'Employment Length',
 'Policy Code']

In [11]:
columns_to_drop = [

    # 1. IDENTIFIERS & ADMIN — carry zero predictive value
    'id',
    'member_id',
    'url',
    'desc',
    'title',
    'zip_code',
    'addr_state',
    'pymnt_plan',
    'policy_code',
    'disbursement_method',
    'application_type',
    'initial_list_status',

    # 2. DATA LEAKAGE — only exist AFTER loan is given
    'funded_amnt',
    'funded_amnt_inv',
    'issue_d',
    'out_prncp',
    'out_prncp_inv',
    'total_pymnt',
    'total_pymnt_inv',
    'total_rec_prncp',
    'total_rec_int',
    'total_rec_late_fee',
    'recoveries',
    'collection_recovery_fee',
    'last_pymnt_d',
    'last_pymnt_amnt',
    'next_pymnt_d',
    'last_credit_pull_d',
    'last_fico_range_high',
    'last_fico_range_low',

    # 3. HARDSHIP COLUMNS — all post-loan information
    'hardship_flag',
    'hardship_type',
    'hardship_reason',
    'hardship_status',
    'deferral_term',
    'hardship_amount',
    'hardship_start_date',
    'hardship_end_date',
    'payment_plan_start_date',
    'hardship_length',
    'hardship_dpd',
    'hardship_loan_status',
    'orig_projected_additional_accrued_interest',
    'hardship_payoff_balance_amount',
    'hardship_last_payment_amount',

    # 4. DEBT SETTLEMENT COLUMNS — all post-default information
    'debt_settlement_flag',
    'debt_settlement_flag_date',
    'settlement_status',
    'settlement_date',
    'settlement_amount',
    'settlement_percentage',
    'settlement_term',

    # 5. JOINT APPLICATION COLUMNS — mostly empty, rarely used
    'annual_inc_joint',
    'dti_joint',
    'verification_status_joint',
    'revol_bal_joint',
    'sec_app_fico_range_low',
    'sec_app_fico_range_high',
    'sec_app_earliest_cr_line',
    'sec_app_inq_last_6mths',
    'sec_app_mort_acc',
    'sec_app_open_acc',
    'sec_app_revol_util',
    'sec_app_open_act_il',
    'sec_app_num_rev_accts',
    'sec_app_chargeoff_within_12_mths',
    'sec_app_collections_12_mths_ex_med',
    'sec_app_mths_since_last_major_derog',

    # 6. REDUNDANT COLUMNS — better version already exists
    'sub_grade',          # grade already covers this
    'emp_title',          # too many unique text values, unusable
    'fico_range_high',    # keep fico_range_low only, they're almost identical
]

In [13]:
df_acc = df_acc.drop(columns=columns_to_drop, errors='ignore')

In [18]:
df_acc.columns

Index(['loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'emp_length',
       'home_ownership', 'annual_inc', 'verification_status', 'loan_status',
       'purpose', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low',
       'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record',
       'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
       'collections_12_mths_ex_med', 'mths_since_last_major_derog',
       'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'open_acc_6m',
       'open_act_il', 'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il',
       'total_bal_il', 'il_util', 'open_rv_12m', 'open_rv_24m', 'max_bal_bc',
       'all_util', 'total_rev_hi_lim', 'inq_fi', 'total_cu_tl', 'inq_last_12m',
       'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util',
       'chargeoff_within_12_mths', 'delinq_amnt', 'mo_sin_old_il_acct',
       'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl',
       'mort_acc', 'mths_sin

In [20]:
columns_to_drop_2 = [

    # REDUNDANT UTILIZATION — dti & revol_util already cover this
    'all_util',
    'il_util',
    'bc_util',

    # REDUNDANT ACCOUNT COUNTS — open_acc & total_acc already cover this
    'open_acc_6m',
    'open_act_il',
    'open_il_12m',
    'open_il_24m',
    'open_rv_12m',
    'open_rv_24m',
    'num_sats',
    'num_actv_bc_tl',
    'num_bc_sats',
    'num_bc_tl',
    'num_il_tl',
    'num_op_rev_tl',
    'num_rev_accts',
    'num_rev_tl_bal_gt_0',
    'num_tl_op_past_12m',
    'acc_open_past_24mths',
    'total_cu_tl',

    # REDUNDANT BALANCES — tot_cur_bal already covers this
    'total_bal_il',
    'avg_cur_bal',
    'max_bal_bc',
    'total_bal_ex_mort',

    # REDUNDANT CREDIT LIMITS
    'total_bc_limit',
    'total_il_high_credit_limit',
    'bc_open_to_buy',

    # REDUNDANT TIMING — too many "months since" columns
    'mths_since_rcnt_il',
    'mths_since_recent_bc',
    'mths_since_recent_inq',
    'mo_sin_old_il_acct',
    'mo_sin_old_rev_tl_op',
    'mo_sin_rcnt_rev_tl_op',
    'mo_sin_rcnt_tl',

    # REDUNDANT INQUIRIES — inq_last_6mths already covers this
    'inq_fi',
    'inq_last_12m',
]

df_acc = df_acc.drop(columns=columns_to_drop_2, errors='ignore')
print(f"Columns remaining: {len(df_acc.columns)}")

Columns remaining: 44


In [21]:
missing = df_acc.isnull().sum()

In [22]:
missing

loan_amnt                              33
term                                   33
int_rate                               33
installment                            33
grade                                  33
emp_length                         146940
home_ownership                         33
annual_inc                             37
verification_status                    33
loan_status                            33
purpose                                33
dti                                  1744
delinq_2yrs                            62
earliest_cr_line                       62
fico_range_low                         33
inq_last_6mths                         63
mths_since_last_delinq            1158535
mths_since_last_record            1901545
open_acc                               62
pub_rec                                62
revol_bal                              33
revol_util                           1835
total_acc                              62
collections_12_mths_ex_med        

In [23]:
len(df_acc)

2260701

In [25]:
df = df_acc

In [26]:
# ── STEP 1 — Drop rows with tiny nulls ──────────────────────────
tiny_null_cols = [
    'loan_amnt', 'term', 'int_rate', 'installment', 'grade',
    'home_ownership', 'annual_inc', 'verification_status',
    'loan_status', 'purpose', 'revol_bal', 'fico_range_low',
    'delinq_2yrs', 'earliest_cr_line', 'inq_last_6mths',
    'open_acc', 'pub_rec', 'total_acc', 'acc_now_delinq',
    'delinq_amnt'
]
before = len(df)
df = df.dropna(subset=tiny_null_cols)
after = len(df)
print(f"Step 1 done — Dropped {before - after} rows")

# ── STEP 2 — Drop columns with >60% nulls ───────────────────────
high_null_cols = [
    'mths_since_last_record',          # 84% null
    'mths_since_recent_bc_dlq',        # 77% null
    'mths_since_last_major_derog',     # 74% null
    'mths_since_recent_revol_delinq'   # 67% null
]
df = df.drop(columns=high_null_cols, errors='ignore')
print(f"Step 2 done — Dropped {len(high_null_cols)} columns")

# ── STEP 3 — Fill emp_length ─────────────────────────────────────
df['emp_length'] = df['emp_length'].fillna('< 1 year')
print(f"Step 3 done — emp_length filled")

# ── STEP 4 — Fill with median ────────────────────────────────────
median_fill_cols = [
    'dti', 'revol_util', 'tot_cur_bal',
    'total_rev_hi_lim', 'tot_hi_cred_lim',
    'pct_tl_nvr_dlq', 'percent_bc_gt_75'
]
for col in median_fill_cols:
    if col in df.columns:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
print(f"Step 4 done — Median filled {len(median_fill_cols)} columns")

# ── STEP 5 — Fill with 0 ─────────────────────────────────────────
zero_fill_cols = [
    'tot_coll_amt', 'num_accts_ever_120_pd', 'num_actv_rev_tl',
    'num_tl_30dpd', 'num_tl_90g_dpd_24m', 'num_tl_120dpd_2m',
    'mort_acc', 'pub_rec_bankruptcies', 'tax_liens',
    'collections_12_mths_ex_med', 'chargeoff_within_12_mths'
]
for col in zero_fill_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)
print(f"Step 5 done — Zero filled {len(zero_fill_cols)} columns")

# ── STEP 6 — Fill mths_since_last_delinq with 999 ───────────────
df['mths_since_last_delinq'] = df['mths_since_last_delinq'].fillna(999)
print(f"Step 6 done — mths_since_last_delinq filled with 999")

# ── FINAL CHECK ──────────────────────────────────────────────────
print(f"\n{'='*40}")
print(f"Total rows remaining : {len(df):,}")
print(f"Total columns        : {len(df.columns)}")
print(f"Nulls remaining      : {df.isnull().sum().sum()}")
print(f"{'='*40}")

Step 1 done — Dropped 63 rows
Step 2 done — Dropped 4 columns
Step 3 done — emp_length filled
Step 4 done — Median filled 7 columns
Step 5 done — Zero filled 11 columns
Step 6 done — mths_since_last_delinq filled with 999

Total rows remaining : 2,260,638
Total columns        : 40
Nulls remaining      : 0


In [27]:
print(df.columns.tolist())
print(f"\nData types:")
print(df.dtypes)

['loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'loan_status', 'purpose', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'inq_last_6mths', 'mths_since_last_delinq', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'collections_12_mths_ex_med', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim', 'chargeoff_within_12_mths', 'delinq_amnt', 'mort_acc', 'num_accts_ever_120_pd', 'num_actv_rev_tl', 'num_tl_120dpd_2m', 'num_tl_30dpd', 'num_tl_90g_dpd_24m', 'pct_tl_nvr_dlq', 'percent_bc_gt_75', 'pub_rec_bankruptcies', 'tax_liens', 'tot_hi_cred_lim']

Data types:
loan_amnt                     float64
term                           object
int_rate                      float64
installment                   float64
grade                          object
emp_length                     object
home_ownership                 object
annual_inc                    float64
verification_

In [28]:
# Check what values exist in loan_status
print(df['loan_status'].value_counts())

loan_status
Fully Paid                                             1076750
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1962
Does not meet the credit policy. Status:Charged Off        758
Default                                                     40
Name: count, dtype: int64


In [29]:
# Remove loans that are still active
remove_statuses = ['Current', 'In Grace Period']
df = df[~df['loan_status'].isin(remove_statuses)]

print(f"Rows after removing active loans: {len(df):,}")

# Create binary target variable
# 0 = Good borrower
# 1 = Defaulted / High Risk
def assign_target(status):
    if status in ['Fully Paid',
                  'Does not meet the credit policy. Status:Fully Paid']:
        return 0
    else:
        return 1

df['default'] = df['loan_status'].apply(assign_target)

# Drop original loan_status — no longer needed
df.drop('loan_status', axis=1, inplace=True)

# Check result
print(f"\nTarget distribution:")
print(df['default'].value_counts())
print(f"\nDefault rate: {df['default'].mean():.2%}")
print(f"\nColumns remaining: {len(df.columns)}")

Rows after removing active loans: 1,373,885

Target distribution:
default
0    1078712
1     295173
Name: count, dtype: int64

Default rate: 21.48%

Columns remaining: 40


In [30]:
# Check them quickly
object_cols = df.select_dtypes(include='object').columns.tolist()
print(object_cols)

# Check unique values in each
for col in object_cols:
    print(f"\n{col}:")
    print(df[col].unique())

['term', 'grade', 'emp_length', 'home_ownership', 'verification_status', 'purpose', 'earliest_cr_line']

term:
[' 36 months' ' 60 months']

grade:
['C' 'B' 'F' 'A' 'E' 'D' 'G']

emp_length:
['10+ years' '3 years' '4 years' '6 years' '7 years' '8 years' '2 years'
 '5 years' '9 years' '< 1 year' '1 year']

home_ownership:
['MORTGAGE' 'RENT' 'OWN' 'ANY' 'NONE' 'OTHER']

verification_status:
['Not Verified' 'Source Verified' 'Verified']

purpose:
['debt_consolidation' 'small_business' 'home_improvement' 'major_purchase'
 'credit_card' 'other' 'house' 'vacation' 'car' 'medical' 'moving'
 'renewable_energy' 'wedding' 'educational']

earliest_cr_line:
['Aug-2003' 'Dec-1999' 'Aug-2000' 'Jun-1998' 'Oct-1987' 'Jun-1990'
 'Feb-1999' 'Apr-2002' 'Nov-1994' 'Jun-1996' 'Jun-2005' 'May-1984'
 'Dec-2001' 'Nov-1993' 'Mar-2005' 'May-2004' 'Jun-1991' 'May-2000'
 'Oct-2011' 'May-1994' 'Jul-2011' 'May-1991' 'May-2001' 'Jun-2002'
 'Dec-1985' 'Apr-2007' 'Feb-2002' 'Jun-2001' 'Oct-1996' 'Jan-2005'
 'Jul-2001' 

In [31]:
df['term'] = df['term'].str.extract('(\d+)').astype(int)

print("term done:")
print(df['term'].value_counts())

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_57/2828373808.py:1: SyntaxWarning: invalid escape sequence '\d'
  df['term'] = df['term'].str.extract('(\d+)').astype(int)


term done:
term
36    1038196
60     335689
Name: count, dtype: int64


In [32]:
emp_length_map = {
    '< 1 year': 0,
    '1 year'  : 1,
    '2 years' : 2,
    '3 years' : 3,
    '4 years' : 4,
    '5 years' : 5,
    '6 years' : 6,
    '7 years' : 7,
    '8 years' : 8,
    '9 years' : 9,
    '10+ years': 10
}

df['emp_length'] = df['emp_length'].map(emp_length_map)

print("emp_length done:")
print(df['emp_length'].value_counts().sort_index())

emp_length done:
emp_length
0     191825
1      90569
2     124451
3     110002
4      82430
5      85921
6      63997
7      60646
8      61728
9      51782
10    450534
Name: count, dtype: int64


In [33]:
# A is best (lowest risk), G is worst (highest risk)
grade_map = {
    'A': 1,
    'B': 2,
    'C': 3,
    'D': 4,
    'E': 5,
    'F': 6,
    'G': 7
}

df['grade'] = df['grade'].map(grade_map)

print("grade done:")
print(df['grade'].value_counts().sort_index())

grade done:
grade
1    236845
2    398872
3    391347
4    207386
5     96759
6     33075
7      9601
Name: count, dtype: int64


In [34]:
verification_map = {
    'Not Verified'   : 0,
    'Source Verified': 1,
    'Verified'       : 2
}

df['verification_status'] = df['verification_status'].map(verification_map)

print("verification_status done:")
print(df['verification_status'].value_counts())

verification_status done:
verification_status
1    532456
2    427001
0    414428
Name: count, dtype: int64


In [35]:
# ANY, NONE, OTHER are very rare — group them all as OTHER
df['home_ownership'] = df['home_ownership'].replace({
    'ANY' : 'OTHER',
    'NONE': 'OTHER'
})

print("home_ownership unique values:")
print(df['home_ownership'].value_counts())

home_ownership unique values:
home_ownership
MORTGAGE    678083
RENT        547009
OWN         148257
OTHER          536
Name: count, dtype: int64


In [36]:
# One-hot encode purpose (14 categories, no natural order)
df = pd.get_dummies(df, columns=['purpose'], prefix='purpose', drop_first=True)

print("purpose one-hot encoded")
print(f"Columns now: {len(df.columns)}")

purpose one-hot encoded
Columns now: 52


In [37]:
# Now encode home_ownership
home_map = {
    'MORTGAGE': 0,
    'RENT'    : 1,
    'OWN'     : 2,
    'OTHER'   : 3
}

df['home_ownership'] = df['home_ownership'].map(home_map)

print("home_ownership done:")
print(df['home_ownership'].value_counts())

home_ownership done:
home_ownership
0    678083
1    547009
2    148257
3       536
Name: count, dtype: int64


In [38]:
from datetime import datetime

# Convert to datetime
df['earliest_cr_line'] = pd.to_datetime(
    df['earliest_cr_line'],
    format='%b-%Y',
    errors='coerce'
)

# Convert to years of credit history
reference_date = datetime(2025, 1, 1)
df['credit_history_years'] = (
    reference_date - df['earliest_cr_line']
).dt.days / 365.25

# Drop original column
df.drop('earliest_cr_line', axis=1, inplace=True)

print("credit_history_years done:")
print(df['credit_history_years'].describe())

credit_history_years done:
count    1.373885e+06
mean     2.579091e+01
std      7.623297e+00
min      9.253936e+00
25%      2.050376e+01
50%      2.441889e+01
75%      2.967283e+01
max      9.075428e+01
Name: credit_history_years, dtype: float64


In [39]:
# Confirm no object columns remain
print("Remaining object columns:")
print(df.select_dtypes(include='object').columns.tolist())

print(f"\nTotal columns: {len(df.columns)}")
print(f"Total rows: {len(df):,}")
print(f"\nAny nulls: {df.isnull().sum().sum()}")
print(f"\nData types:")
print(df.dtypes.value_counts())

Remaining object columns:
[]

Total columns: 52
Total rows: 1,373,885

Any nulls: 0

Data types:
float64    33
bool       13
int64       6
Name: count, dtype: int64


In [40]:
# Convert bool columns to int (0 and 1)
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

print("Bool columns converted to int")
print(df.dtypes.value_counts())

Bool columns converted to int
float64    33
int64      19
Name: count, dtype: int64


In [41]:
# Separate features and target
X = df.drop('default', axis=1)
y = df['default']

# Check imbalance
print(f"Total rows     : {len(y):,}")
print(f"Good loans (0) : {(y==0).sum():,} ({(y==0).mean():.2%})")
print(f"Defaulted  (1) : {(y==1).sum():,} ({(y==1).mean():.2%})")
print(f"\nImbalance ratio: {(y==0).sum() / (y==1).sum():.1f}:1")

Total rows     : 1,373,885
Good loans (0) : 1,078,712 (78.52%)
Defaulted  (1) : 295,173 (21.48%)

Imbalance ratio: 3.7:1


In [42]:
# Calculate scale_pos_weight
scale_pos_weight = (y==0).sum() / (y==1).sum()
print(f"scale_pos_weight = {scale_pos_weight:.2f}")

# Output: scale_pos_weight = 3.65

scale_pos_weight = 3.65


In [43]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 80% train, 20% test
    random_state=42,     # reproducible results
    stratify=y           # maintain same imbalance ratio in both splits
)

print(f"Training rows  : {len(X_train):,}")
print(f"Testing rows   : {len(X_test):,}")
print(f"\nTraining target distribution:")
print(y_train.value_counts())
print(f"\nTesting target distribution:")
print(y_test.value_counts())

Training rows  : 1,099,108
Testing rows   : 274,777

Training target distribution:
default
0    862970
1    236138
Name: count, dtype: int64

Testing target distribution:
default
0    215742
1     59035
Name: count, dtype: int64


In [44]:
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             roc_auc_score, confusion_matrix)
import time

# Define model with all important parameters
model = XGBClassifier(
    n_estimators      = 200,      # number of trees
    max_depth         = 6,        # depth of each tree
    learning_rate     = 0.1,      # how fast it learns
    scale_pos_weight  = 3.65,     # handles class imbalance
    use_label_encoder = False,
    eval_metric       = 'auc',
    random_state      = 42,
    n_jobs            = -1        # use all CPU cores
)

# Train the model
print("Training started...")
start = time.time()

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50            # print progress every 50 trees
)

end = time.time()
print(f"\nTraining completed in {(end-start)/60:.1f} minutes")

Training started...
[0]	validation_0-auc:0.69939


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [08:52:38] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[50]	validation_0-auc:0.71821
[100]	validation_0-auc:0.72238
[150]	validation_0-auc:0.72454
[199]	validation_0-auc:0.72591

Training completed in 0.4 minutes


In [45]:
# Make predictions
y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
auc      = roc_auc_score(y_test, y_pred_prob)

print("=" * 40)
print(f"Accuracy : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"AUC-ROC  : {auc:.4f}")
print("=" * 40)

print("\nDetailed Report:")
print(classification_report(y_test, y_pred,
      target_names=['Good Loan', 'Default']))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy : 0.6573 (65.73%)
AUC-ROC  : 0.7259

Detailed Report:
              precision    recall  f1-score   support

   Good Loan       0.88      0.65      0.75    215742
     Default       0.35      0.68      0.46     59035

    accuracy                           0.66    274777
   macro avg       0.61      0.66      0.60    274777
weighted avg       0.77      0.66      0.69    274777


Confusion Matrix:
[[140598  75144]
 [ 19032  40003]]


In [46]:
import pickle

with open('xgb_model.pkl', 'wb') as f:
    pickle.dump(model, f)

# Save feature names too — needed for SHAP later
with open('feature_names.pkl', 'wb') as f:
    pickle.dump(list(X.columns), f)

print("Model saved successfully")
print(f"Features saved: {len(X.columns)}")

Model saved successfully
Features saved: 51


In [47]:
print(f"Accuracy : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"AUC-ROC  : {auc:.4f}")

Accuracy : 0.6573 (65.73%)
AUC-ROC  : 0.7259


In [48]:
model_v2 = XGBClassifier(
    n_estimators     = 300,
    max_depth        = 5,
    learning_rate    = 0.05,
    scale_pos_weight = 1.5,    # ← Reduced from 3.65 to 1.5
    subsample        = 0.8,    # ← Use 80% of data per tree
    colsample_bytree = 0.8,    # ← Use 80% of features per tree
    min_child_weight = 5,      # ← Prevents overfitting
    use_label_encoder = False,
    eval_metric      = 'auc',
    random_state     = 42,
    n_jobs           = -1
)

model_v2.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

# Evaluate
y_pred_v2      = model_v2.predict(X_test)
y_pred_prob_v2 = model_v2.predict_proba(X_test)[:, 1]

accuracy_v2 = accuracy_score(y_test, y_pred_v2)
auc_v2      = roc_auc_score(y_test, y_pred_prob_v2)

print("=" * 40)
print(f"V1 Accuracy : 66.02%")
print(f"V2 Accuracy : {accuracy_v2*100:.2f}%")
print("=" * 40)
print(f"AUC-ROC V2  : {auc_v2:.4f}")

print("\nDetailed Report V2:")
print(classification_report(
    y_test, y_pred_v2,
    target_names=['Good Loan', 'Default']
))

print("\nConfusion Matrix V2:")
print(confusion_matrix(y_test, y_pred_v2))

[0]	validation_0-auc:0.69587


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [08:54:45] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[50]	validation_0-auc:0.71079
[100]	validation_0-auc:0.71648
[150]	validation_0-auc:0.71895
[200]	validation_0-auc:0.72079
[250]	validation_0-auc:0.72211
[299]	validation_0-auc:0.72314
V1 Accuracy : 66.02%
V2 Accuracy : 78.45%
AUC-ROC V2  : 0.7231

Detailed Report V2:
              precision    recall  f1-score   support

   Good Loan       0.82      0.93      0.87    215742
     Default       0.50      0.24      0.33     59035

    accuracy                           0.78    274777
   macro avg       0.66      0.59      0.60    274777
weighted avg       0.75      0.78      0.75    274777


Confusion Matrix V2:
[[201213  14529]
 [ 44685  14350]]


In [49]:
import pickle

# Save model
with open('xgb_model.pkl', 'wb') as f:
    pickle.dump(model, f)

# Save feature names
with open('feature_names.pkl', 'wb') as f:
    pickle.dump(list(X.columns), f)

# OPTIONAL:
# Save scaler/encoders if used
# pickle.dump(scaler, open('scaler.pkl', 'wb'))

print("Model and metadata saved successfully")
print(f"Total features: {len(X.columns)}")

Model and metadata saved successfully
Total features: 51
